# LinearRegression baseline (YLDBE)

**Purpose:** Same feature matrix, train/test framing, and **grouped OOF logic** as the GradientBoosting / HistGB pipeline in `maize_gxe_ml_pipeline.py`: GPC from **PCA refit inside each GroupKFold fold** on training lines, plus `env_*`, `past_yld_*`, and `GXE__*` interaction columns.

**Why this often underperforms GBR:** `LinearRegression` assumes a **linear** response in the features. GxE and weather effects are typically **non-linear** and **state-dependent**; tree ensembles approximate interactions and thresholds without explicit products. Hand-built `GPC × env` products help linear models but still miss higher-order and non-additive structure that boosting captures.

**Note:** OOF predictions use `grouped_oof_with_pca` (not `cross_val_predict` on a fixed design matrix), matching the GBR OOF path exactly.

## 1. Imports & load data

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")

# Repo root (notebook may run with cwd = project root or output/)
_here = Path.cwd().resolve()
ROOT = _here if (_here / "maize_gxe_ml_pipeline.py").exists() else _here.parent
if not (ROOT / "maize_gxe_ml_pipeline.py").exists() and _here.name.lower() == "output":
    ROOT = _here.parent
sys.path.insert(0, str(ROOT))

from maize_gxe_ml_pipeline import (
    fit_pca_for_lines,
    load_config,
    map_pca_to_rows,
    metrics_dict,
    step5_environment_features,
    step6_select_env_and_gxe,
)

CFG_PATH = ROOT / "pipeline_config.yaml"
cfg = load_config(CFG_PATH)
out = Path(cfg["output_dir"])
if not out.is_absolute():
    out = (CFG_PATH.parent / out).resolve()

cohort = str(cfg["cohort"]).upper()
tr_min, tr_max = int(cfg["years"]["train_min"]), int(cfg["years"]["train_max"])
te_year = int(cfg["years"]["test_year"])
pca_n = int(cfg["hypers"]["pca_components"])
n_splits = int(cfg["hypers"]["group_cv_splits"])
rs = int(cfg.get("random_state", 42))
rng = np.random.default_rng(rs)

full_path = out / "full_dataset.csv"
geno_path = out / f"genotypes_{cohort.lower()}.csv"
assert full_path.exists(), f"Missing {full_path} — run the main pipeline first."
assert geno_path.exists(), f"Missing {geno_path} — run the main pipeline first."

full_dataset = pd.read_csv(full_path, low_memory=False)
geno = pd.read_csv(geno_path, low_memory=False)
snp_cols = [c for c in geno.columns if c != "line_id"]

print("output_dir:", out)
print("full_dataset:", full_dataset.shape)
print("geno:", geno.shape)
print("cohort:", cohort, "| train years:", tr_min, "-", tr_max, "| test year:", te_year)


## 2. Preprocessing (match GBR modeling frame)

- Rebuild the same `model_df` slice: train-year rows + test year, lines with genotypes, optional subsampling (`max_train_rows`, `max_rows_per_env`).
- **Environment:** `step5_environment_features` (global `StandardScaler` on numeric env block).
- **Genetics:** PCA on SNPs (50 components by default); GPC columns appended per row.
- **GxE / env subset:** Prefer column list from `env_gxe_feature_columns.csv` (exact GBR order). If missing, rerun `step6_select_env_and_gxe` on the train slice.
- **Linear model:** Trees (GBR) are scale-invariant; **linear models benefit from an extra `StandardScaler` on the full design** `[GPC | env | past | GxE]` inside each CV fold and for the final fit (see next cells).
- **NaNs:** Same as pipeline: numeric features coerced and **filled with 0** after assembly. Optional strict row filter `COMPLETENESS_MIN` (default `0` = **strict parity** with GBR; set to `0.5` to drop rows with >50% missing features among `feature_cols`).

In [ ]:
train_year_mask = full_dataset["YEAR"].between(tr_min, tr_max) & full_dataset["YLDBE"].notna()
test_year_mask = (full_dataset["YEAR"] == te_year) & full_dataset["YLDBE"].notna()
model_df = full_dataset.loc[train_year_mask | test_year_mask].copy()
model_df = model_df[model_df["line_id"].isin(set(geno["line_id"]))].copy()

max_rows = cfg.get("max_train_rows")
per_env = cfg.get("max_rows_per_env")
train_only = model_df.loc[model_df["YEAR"].between(tr_min, tr_max)]
if per_env:
    parts = []
    for env_id, chunk in train_only.groupby("env_id"):
        if len(chunk) > int(per_env):
            parts.append(chunk.sample(n=int(per_env), random_state=int(rng.integers(0, 1_000_000))))
        else:
            parts.append(chunk)
    train_only = pd.concat(parts, axis=0)
if max_rows is not None and len(train_only) > int(max_rows):
    train_only = train_only.sample(n=int(max_rows), random_state=42)
test_only = model_df.loc[model_df["YEAR"] == te_year]
model_df = pd.concat([train_only, test_only], axis=0).drop_duplicates()

snp_in_m = [c for c in snp_cols if c in model_df.columns]
if snp_in_m:
    model_df = model_df.drop(columns=snp_in_m, errors="ignore")

model_df, env_numeric, _env_scaler = step5_environment_features(
    model_df, str(cfg.get("env_scaling", "global"))
)

train_lines = model_df.loc[model_df["YEAR"].between(tr_min, tr_max), "line_id"].unique()
_, final_pca_pipe = fit_pca_for_lines(geno, snp_cols, train_lines, pca_n, rs)
gpc_df = map_pca_to_rows(geno, snp_cols, model_df["line_id"], final_pca_pipe)
gpc_cols = list(gpc_df.columns)
for c in gpc_cols:
    model_df[c] = gpc_df[c].values

is_train = model_df["YEAR"].between(tr_min, tr_max).values
env_only_numeric = [
    c for c in env_numeric if c in model_df.columns and not str(c).startswith("GPC_")
]

engxe_path = out / "env_gxe_feature_columns.csv"
if engxe_path.exists():
    nongeno_feature_cols = pd.read_csv(engxe_path)["env_and_gxe_column"].astype(str).tolist()
    for c in nongeno_feature_cols:
        if c in model_df.columns:
            continue
        if c.startswith("GXE__") and "__x__" in c:
            rest = c[len("GXE__") :]
            gc, ec = rest.split("__x__", 1)
            model_df[c] = model_df[gc].astype(float) * model_df[ec].astype(float)
        else:
            raise KeyError(f"Column {c!r} missing from model_df; rerun main pipeline.")
else:
    past_cols = [c for c in ("past_yld_mean", "past_yld_median") if c in model_df.columns]
    env_chosen, pair_list = step6_select_env_and_gxe(
        model_df.loc[is_train].copy(),
        env_only_numeric,
        gpc_cols,
        rng,
        int(cfg.get("max_env_features_gxe", 20)),
        int(cfg.get("n_interaction_pairs", 10)),
    )
    interact_cols = []
    for gc, ec in pair_list:
        cname = f"GXE__{gc}__x__{ec}"
        model_df[cname] = model_df[gc].astype(float) * model_df[ec].astype(float)
        interact_cols.append(cname)
    nongeno_feature_cols = env_chosen + past_cols + interact_cols
    nongeno_feature_cols = [c for c in nongeno_feature_cols if c in model_df.columns]

feature_cols = gpc_cols + nongeno_feature_cols
feature_cols = [c for c in feature_cols if c in model_df.columns]

# Optional completeness filter (0 = parity with GBR fillna(0) approach)
COMPLETENESS_MIN = 0.0
if COMPLETENESS_MIN > 0:
    frac = model_df[feature_cols].notna().sum(axis=1) / len(feature_cols)
    model_df = model_df.loc[frac >= COMPLETENESS_MIN].copy()

model_df = model_df.reset_index(drop=True)
is_train = model_df["YEAR"].between(tr_min, tr_max).values

G = model_df[gpc_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
Ng = model_df[nongeno_feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
X_all = np.hstack([G, Ng])
y_all = model_df["YLDBE"].values.astype(float)
groups_all = model_df["env_id"].astype(str).values
train_idx = is_train

assert X_all.shape[0] == len(model_df), "X rows must match modeling frame"
assert X_all.shape[0] <= len(full_dataset), "Modeling frame should be subset of full_dataset"

# Sanity vs saved PCA artifact (optional)
pca_pkl = out / "pca_pipeline.pkl"
if pca_pkl.exists():
    pipe_disk = joblib.load(pca_pkl)
    gpc_disk = map_pca_to_rows(geno, snp_cols, model_df["line_id"], pipe_disk).values
    max_abs = np.nanmax(np.abs(G - gpc_disk))
    print("max |GPC - pca_pipeline.pkl transform|:", max_abs)
    if max_abs > 1e-3:
        print("Note: small drift if train_lines differ from prior run.")

summary_feats = pd.DataFrame(
    {
        "prefix": [c.split("_")[0][:6] for c in feature_cols],
        "feature": feature_cols,
    }
)
print("Target: YLDBE | n rows:", len(model_df), "| n features:", len(feature_cols))
print("Groups (env_id) unique:", pd.Series(groups_all).nunique())
display(summary_feats.groupby("prefix").size().to_frame("n"))
display(pd.Series(feature_cols, name="feature").to_frame().head(25).style.set_caption("Features (head)"))


## 3. Model training & grouped CV (PCA refit per fold)

`GroupKFold` groups by **`env_id`** so lines in the same environment stay in train or test together.

**Same OOF logic as GBR:** for each fold we refit PCA on **training lines only**, map GPC for train/test rows, stack with `nongeno_feature_cols`, then **`StandardScaler` on the full design** (extra vs GBR; linear models need comparable feature scales) and fit `LinearRegression`.

`cross_val_predict(LinearRegression, X, y, ...)` on a **single** precomputed `X` would **not** refit PCA inside folds — that would **not** match the GBR notebook/pipeline; we therefore use an explicit loop mirroring `grouped_oof_with_pca`.

In [ ]:
def grouped_oof_lr_with_pca(
    df, snp_cols, geno_wide, nongeno_feature_cols, y, groups, n_splits, pca_n, random_state
):
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.full(len(y), np.nan)
    fold_metrics = []
    for tr, te in gkf.split(df, y, groups=groups):
        tr_lines = df.iloc[tr]["line_id"].unique()
        _, pipe = fit_pca_for_lines(geno_wide, snp_cols, tr_lines, pca_n, random_state)
        gpc_tr = map_pca_to_rows(geno_wide, snp_cols, df.iloc[tr]["line_id"], pipe).values
        gpc_te = map_pca_to_rows(geno_wide, snp_cols, df.iloc[te]["line_id"], pipe).values
        Xtr = np.hstack([gpc_tr, df.iloc[tr][nongeno_feature_cols].values.astype(float)])
        Xte = np.hstack([gpc_te, df.iloc[te][nongeno_feature_cols].values.astype(float)])
        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)
        lr = LinearRegression(fit_intercept=True)
        lr.fit(Xtr_s, y[tr])
        oof[te] = lr.predict(Xte_s)
        fold_metrics.append(metrics_dict(y[te], oof[te]))
    return oof, fold_metrics


df_train = model_df.loc[train_idx].reset_index(drop=True)
y_tr = y_all[train_idx]
g_tr = groups_all[train_idx]

oof_train, fold_metrics = grouped_oof_lr_with_pca(
    df_train,
    snp_cols,
    geno,
    nongeno_feature_cols,
    y_tr,
    g_tr,
    n_splits,
    pca_n,
    rs,
)

oof_full = np.full(len(y_all), np.nan)
oof_full[np.where(train_idx)[0]] = oof_train

glob_oof = metrics_dict(y_tr, oof_train)
fold_rmses = [float(fm["rmse"]) for fm in fold_metrics if np.isfinite(fm["rmse"])]
cv_rmse_std = float(np.std(fold_rmses)) if fold_rmses else float("nan")
print("OOF global (train years):", glob_oof)
print("CV RMSE std across folds:", cv_rmse_std)

# Final model: same stacked design as X_all (GPC from train-lines PCA in preprocessing cell)
final_pipe = Pipeline(
    [("scaler", StandardScaler()), ("lr", LinearRegression(fit_intercept=True))]
)
final_pipe.fit(X_all[train_idx], y_all[train_idx])
joblib.dump(final_pipe, out / "lr_model.pkl")
print("Saved", out / "lr_model.pkl")


## 4. Evaluation metrics

OOF metrics on **train-year** rows; **per-`env_id`** table. **Expect higher RMSE than GBR** when GxE is non-linear: the linear model misses curvature and high-order interactions that boosting fits.

Compare to `run_summary.json` → `oof_global_metrics` when present.

In [ ]:
glob_oof = metrics_dict(y_tr, oof_train)
metrics_global = pd.DataFrame([glob_oof]).T
metrics_global.columns = ["value"]
display(metrics_global.style.format("{:.4f}").set_caption("OOF global (LinearRegression)"))

by_env_rows = []
tr_df = model_df.loc[train_idx]
for eid, chunk in tr_df.groupby("env_id"):
    pred_c = oof_full[chunk.index.values]
    yt = chunk["YLDBE"].values
    m = metrics_dict(yt, pred_c)
    m["env_id"] = eid
    m["n"] = len(chunk)
    by_env_rows.append(m)
metrics_by_env = pd.DataFrame(by_env_rows).sort_values("rmse", na_position="last")
display(
    metrics_by_env.head(20).style.format({"rmse": "{:.3f}", "mae": "{:.3f}", "r2": "{:.3f}", "n": "{:.0f}"}).set_caption(
        "Per-env OOF (head)"
    )
)

gbr_row = None
summary_path = out / "run_summary.json"
if summary_path.exists():
    with open(summary_path, encoding="utf-8") as f:
        summ = json.load(f)
    gbr_oof = summ.get("oof_global_metrics", {})
    gbr_row = pd.DataFrame(
        {
            "model": ["GBR (OOF)", "LinearRegression (OOF)"],
            "rmse": [gbr_oof.get("rmse"), glob_oof["rmse"]],
            "mae": [gbr_oof.get("mae"), glob_oof["mae"]],
            "r2": [gbr_oof.get("r2"), glob_oof["r2"]],
        }
    )
    display(gbr_row.style.format({"rmse": "{:.4f}", "mae": "{:.4f}", "r2": "{:.4f}"}).set_caption("GBR vs Linear (global OOF)"))
else:
    print("No run_summary.json — run main pipeline for GBR comparison.")

fig, ax = plt.subplots(figsize=(7, 5))
res = tr_df["YLDBE"].values - oof_train
ax.axhline(0, color="gray", lw=0.8)
ax.scatter(oof_train, res, alpha=0.2, s=10, edgecolors="none")
ax.set_xlabel("OOF predicted YLDBE")
ax.set_ylabel("Residual (obs − pred)")
ax.set_title("LinearRegression — residuals vs OOF prediction (train years)")
fig.tight_layout()
fig.savefig(out / "lr_residuals.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(tr_df["YLDBE"].values, oof_train, alpha=0.2, s=10, edgecolors="none")
lims = [
    min(tr_df["YLDBE"].min(), oof_train.min()),
    max(tr_df["YLDBE"].max(), oof_train.max()),
]
ax.plot(lims, lims, "k--", lw=1, alpha=0.7)
ax.set_xlabel("Observed YLDBE")
ax.set_ylabel("OOF predicted YLDBE")
ax.set_title("Pred vs true (train-year OOF)")
ax.set_aspect("equal", adjustable="box")
fig.tight_layout()
fig.savefig(out / "lr_pred_true.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(10, max(4, 0.2 * len(metrics_by_env))))
sub = metrics_by_env.dropna(subset=["r2"]).sort_values("r2")
sub = sub.assign(env_id=sub["env_id"].astype(str))
sns.barplot(data=sub, y="env_id", x="r2", ax=ax, color="steelblue")
ax.set_title("Per-environment OOF R² (LinearRegression)")
fig.tight_layout()
fig.savefig(out / "lr_by_env_r2.png", dpi=150)
plt.show()


## 5. Predictions & ranking (test year / holdout envs)

Same convention as the main pipeline: **top-k lines per `env_id`** on **test-year** rows using **final** `Pipeline` predictions. Uncertainty proxy: **global OOF RMSE** (and fold RMSE spread in `lr_summary.json`). Save rankings and OOF table for comparison with GBR `rankings_topk_per_env.csv` / `oof_preds.csv`.

In [ ]:
topk = int(cfg.get("topk", 10))
te_mask = model_df["YEAR"].values == te_year
pred_all = final_pipe.predict(X_all)

oof_export = model_df[
    ["line_id", "env_id", "YEAR", "LOC", "YLDBE"]
].copy()
oof_export["oof_pred_lr"] = oof_full
oof_export["pred_final_lr"] = pred_all
oof_export.to_csv(out / "lr_oof_preds.csv", index=False)

rank_rows = []
for env_id, chunk in model_df.loc[te_mask].groupby("env_id"):
    order = np.argsort(-pred_all[chunk.index.values])
    top_idx = chunk.index.values[order[:topk]]
    sub = model_df.loc[top_idx].copy()
    sub["pred_final_lr"] = pred_all[top_idx]
    sub["rank"] = np.arange(1, len(sub) + 1)
    sub["uncertainty_oof_rmse"] = glob_oof["rmse"]
    sub["cv_rmse_std"] = cv_rmse_std
    rank_rows.append(sub)
lr_rank = pd.concat(rank_rows, axis=0) if rank_rows else pd.DataFrame()
lr_rank.to_csv(out / "lr_rankings_topk_per_env.csv", index=False)
print("Saved rankings:", out / "lr_rankings_topk_per_env.csv")

gbr_path = out / "rankings_topk_per_env.csv"
if gbr_path.exists() and len(lr_rank):
    gbr_r = pd.read_csv(gbr_path)
    # overlap: line_id + env_id in top-k both models
    lr_keys = set(zip(lr_rank["line_id"], lr_rank["env_id"]))
    gbr_keys = set(zip(gbr_r["line_id"], gbr_r["env_id"]))
    inter = lr_keys & gbr_keys
    print("Top-k overlap (line, env) LR ∩ GBR:", len(inter), "/", len(lr_keys))
else:
    print("GBR rankings not found or empty LR rank — skip overlap.")

display(lr_rank.head(12).style.format({"pred_final_lr": "{:.2f}", "YLDBE": "{:.2f}"}).set_caption("LR top-k (head)"))


## 6. Interpretation — coefficients

**Linear coefficients** are easy to read (sign/magnitude on standardized features) but **do not encode non-linear interactions** the way tree ensembles do; GBR can assign importance to **regions** of the feature space, not just global slopes.

Bucket features like the pipeline: Genetic (GPC), GxE (`GXE__`), Weather, Soil, Past phenotype, Other.

In [ ]:
lr = final_pipe.named_steps["lr"]
coef = lr.coef_
feat_df = pd.DataFrame({"feat": feature_cols, "coef": coef})
feat_df["abs_coef"] = feat_df["coef"].abs()
feat_df = feat_df.sort_values("abs_coef", ascending=False)


def bucket(name: str) -> str:
    if name.startswith("GPC_"):
        return "Genetic"
    if name.startswith("GXE__"):
        return "GxE"
    if any(x in name.upper() for x in ("PRCP", "TAVG", "HTDD", "CLDD", "DP01", "DP10")):
        return "Weather"
    if any(x in name.upper() for x in ("CLAY", "SILT", "SAND", "PH", "SOC", "NITROGEN", "CFVO")):
        return "Soil"
    if "past_yld" in name:
        return "Past_pheno"
    return "Other"


feat_df["bucket"] = feat_df["feat"].map(bucket)
top20 = feat_df.head(20)

fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=top20, y="feat", x="coef", hue="bucket", dodge=False, ax=ax)
ax.axvline(0, color="gray", lw=0.8)
ax.set_title("Top 20 |coef| (standardized design)")
fig.tight_layout()
fig.savefig(out / "lr_coef_plot.png", dpi=150, bbox_inches="tight")
plt.show()

bucket_sum = feat_df.groupby("bucket")["abs_coef"].sum().reset_index()
display(bucket_sum.style.format({"abs_coef": "{:.4f}"}).set_caption("Sum |coef| by bucket"))


## 7. Summary & config snippet

Log metrics to `lr_summary.json`; print closing line for hackathon narrative.

In [ ]:
summary_path = out / "run_summary.json"
lr_summary = {
    "model": "LinearRegression",
    "oof_global_metrics": glob_oof,
    "cv_fold_rmse": fold_rmses,
    "cv_rmse_std": cv_rmse_std,
    "n_features": int(len(feature_cols)),
    "n_train_rows": int(train_idx.sum()),
    "n_total_rows": int(len(model_df)),
    "group_cv_splits": n_splits,
    "pca_components": pca_n,
    "test_year": te_year,
    "output_files": {
        "model": str(out / "lr_model.pkl"),
        "oof_preds": str(out / "lr_oof_preds.csv"),
        "rankings": str(out / "lr_rankings_topk_per_env.csv"),
        "metrics_by_env": str(out / "lr_metrics_by_env_oof.csv"),
        "plots": [
            str(out / "lr_residuals.png"),
            str(out / "lr_pred_true.png"),
            str(out / "lr_by_env_r2.png"),
            str(out / "lr_coef_plot.png"),
        ],
    },
}

if summary_path.exists():
    with open(summary_path, encoding="utf-8") as f:
        summ = json.load(f)
    lr_summary["gbr_oof_global_metrics"] = summ.get("oof_global_metrics")

def _json_default(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return str(o)


with open(out / "lr_summary.json", "w", encoding="utf-8") as f:
    json.dump(lr_summary, f, indent=2, default=_json_default)

metrics_by_env.to_csv(out / "lr_metrics_by_env_oof.csv", index=False)

yaml_snippet = {
    "linear_regression_baseline": {
        "model": "sklearn.linear_model.LinearRegression",
        "pipeline": ["StandardScaler", "LinearRegression"],
        "group_cv_splits": n_splits,
        "pca_components": pca_n,
        "grouping": "env_id",
        "artifacts": "output/lr_*.csv|json|pkl|png",
    }
}
with open(out / "lr_config_snippet.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_snippet, f, sort_keys=False)

print(yaml.safe_dump(yaml_snippet, sort_keys=False))
print("LinearRegression baseline complete — underperforms GBR expected on GxE non-linearity.")
print("Ideal baseline; GBR superior for hackathon predictions.")
